# GNN-Based Ethereum Phishing/Fraud Detection

**Dataset:** [Ethereum Transactions for Fraud Detection](https://www.kaggle.com/datasets/chaitya0623/ethereum-transactions-for-fraud-detection) (June 2024)

## Pipeline

1. Auto-discover & load dataset files
2. Exploratory Data Analysis
3. Feature Engineering (12 node features from transaction graph)
4. Build DGL graph
5. Train GraphSAGE model (GPU)
6. Evaluate (Precision, Recall, F1, AUC-ROC, AUC-PR)
7. Export model & predictions

---
## 0. Setup & Installation

In [ ]:
# Install DGL (match CUDA version to Kaggle's GPU)
# Check CUDA: !nvcc --version  or  !nvidia-smi
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html -q
# If the above fails, try the CPU fallback:
# !pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html -q

In [ ]:
import os
import glob
import pickle
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

import dgl
from dgl.nn import SAGEConv
from dgl.dataloading import DataLoader, NeighborSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score
)

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Auto-Discover & Load Dataset

This section automatically detects the dataset structure and loads accordingly.
Supports:
- CSV files with transaction edges (`from`, `to`, `value`, ...)
- Pickle files (NetworkX graph)
- Multiple CSV files (edges + nodes + labels)

In [ ]:
# Locate dataset directory
KAGGLE_INPUT = "/kaggle/input/ethereum-transactions-for-fraud-detection"
LOCAL_INPUT = "./data"

DATA_DIR = KAGGLE_INPUT if os.path.exists(KAGGLE_INPUT) else LOCAL_INPUT
print(f"Dataset directory: {DATA_DIR}")

# Discover all files
all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        fp = os.path.join(root, f)
        size_mb = os.path.getsize(fp) / 1e6
        all_files.append((fp, f, size_mb))
        print(f"  {f:50s} {size_mb:10.2f} MB")

print(f"\nTotal files: {len(all_files)}")

In [ ]:
# Preview each CSV file (first 3 rows + columns)
csv_files = [(fp, fn, sz) for fp, fn, sz in all_files if fn.endswith('.csv')]
pkl_files = [(fp, fn, sz) for fp, fn, sz in all_files if fn.endswith('.pkl') or fn.endswith('.pickle')]
parquet_files = [(fp, fn, sz) for fp, fn, sz in all_files if fn.endswith('.parquet')]

dataframes = {}

for fp, fn, sz in csv_files:
    print(f"\n{'='*60}")
    print(f"File: {fn} ({sz:.1f} MB)")
    print('='*60)
    df = pd.read_csv(fp, nrows=5)
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")
    print(df.head(3).to_string())
    dataframes[fn] = fp

for fp, fn, sz in pkl_files:
    print(f"\n{'='*60}")
    print(f"Pickle file: {fn} ({sz:.1f} MB)")
    print('='*60)
    with open(fp, 'rb') as f:
        obj = pickle.load(f)
    print(f"Type: {type(obj).__name__}")
    if hasattr(obj, 'number_of_nodes'):
        print(f"Nodes: {obj.number_of_nodes():,}, Edges: {obj.number_of_edges():,}")

for fp, fn, sz in parquet_files:
    print(f"\n{'='*60}")
    print(f"Parquet file: {fn} ({sz:.1f} MB)")
    print('='*60)
    df = pd.read_parquet(fp, nrows=5) if hasattr(pd, 'read_parquet') else None
    if df is not None:
        print(f"Columns ({len(df.columns)}): {list(df.columns)}")
        print(df.head(3).to_string())

In [ ]:
# =============================================================
# ADAPTIVE DATA LOADER
# =============================================================
# After inspecting the files above, this cell loads the data.
#
# INSTRUCTIONS: Run the cells above first, then adjust the
# column mappings below based on what you see in the preview.
# =============================================================

# --- CONFIGURE THESE BASED ON PREVIEW OUTPUT ---
# If the dataset has a single large CSV with transaction records:
#   Set EDGE_FILE to the filename, and map columns below.
# If it has separate files for edges/nodes/labels:
#   Set each file path accordingly.

# Common column name patterns in Ethereum datasets:
FROM_COL_CANDIDATES = ['from', 'from_address', 'From', 'sender', 'source', 'from_addr']
TO_COL_CANDIDATES = ['to', 'to_address', 'To', 'receiver', 'target', 'to_addr']
VALUE_COL_CANDIDATES = ['value', 'Value', 'amount', 'Amount', 'value_eth', 'ether']
TIMESTAMP_COL_CANDIDATES = ['timestamp', 'Timestamp', 'timeStamp', 'block_timestamp', 'time']
LABEL_COL_CANDIDATES = ['label', 'Label', 'FLAG', 'flag', 'is_fraud', 'isp', 'is_phishing', 'fraud', 'scam']
HASH_COL_CANDIDATES = ['hash', 'Hash', 'tx_hash', 'transaction_hash', 'txHash']

def find_column(df, candidates, required=True):
    """Find the first matching column name from candidates."""
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Could not find column. Tried: {candidates}. Available: {list(df.columns)}")
    return None

print("Ready. Run the next cell after confirming column mappings.")

In [ ]:
%%time

# =============================================================
# LOAD TRANSACTION DATA
# =============================================================
# Pick the largest CSV file as the transaction/edge file
# (adjust if needed based on preview above)

if csv_files:
    # Sort by size descending, pick largest as main transaction file
    main_file = sorted(csv_files, key=lambda x: x[2], reverse=True)[0]
    EDGE_FILE = main_file[0]
    print(f"Loading main file: {main_file[1]} ({main_file[2]:.1f} MB)")
    
    edges_df = pd.read_csv(EDGE_FILE, low_memory=False)
    print(f"Shape: {edges_df.shape}")
    print(f"Columns: {list(edges_df.columns)}")
    print(f"\nFirst 3 rows:")
    print(edges_df.head(3).to_string())

elif pkl_files:
    # Fallback: load pickle file (NetworkX graph)
    print(f"Loading pickle: {pkl_files[0][1]}")
    with open(pkl_files[0][0], 'rb') as f:
        G_nx = pickle.load(f)
    print(f"Nodes: {G_nx.number_of_nodes():,}, Edges: {G_nx.number_of_edges():,}")

else:
    raise FileNotFoundError("No CSV or pickle files found in dataset directory.")

In [ ]:
# Auto-detect column mappings
from_col = find_column(edges_df, FROM_COL_CANDIDATES)
to_col = find_column(edges_df, TO_COL_CANDIDATES)
value_col = find_column(edges_df, VALUE_COL_CANDIDATES, required=False)
ts_col = find_column(edges_df, TIMESTAMP_COL_CANDIDATES, required=False)
label_col = find_column(edges_df, LABEL_COL_CANDIDATES, required=False)
hash_col = find_column(edges_df, HASH_COL_CANDIDATES, required=False)

print(f"=== Detected Column Mapping ===")
print(f"  From address: {from_col}")
print(f"  To address:   {to_col}")
print(f"  Value:        {value_col or 'NOT FOUND (will use 1.0)'}")
print(f"  Timestamp:    {ts_col or 'NOT FOUND (will skip time features)'}")
print(f"  Label:        {label_col or 'NOT FOUND (will check separate file)'}")
print(f"  Tx hash:      {hash_col or 'NOT FOUND'}")

# Check for label in a separate file if not in main file
labels_df = None
if label_col is None:
    for fp, fn, sz in csv_files:
        if fn != os.path.basename(EDGE_FILE):
            temp = pd.read_csv(fp, nrows=5)
            lc = find_column(temp, LABEL_COL_CANDIDATES, required=False)
            if lc:
                print(f"\n  Found labels in separate file: {fn}")
                labels_df = pd.read_csv(fp)
                label_col = lc
                addr_col_in_labels = find_column(labels_df, FROM_COL_CANDIDATES + ['address', 'Address', 'addr'], required=False)
                print(f"  Label column: {label_col}, Address column: {addr_col_in_labels}")
                break

In [ ]:
# Clean and normalize
print("Cleaning data...")

# Normalize addresses to lowercase strings
edges_df[from_col] = edges_df[from_col].astype(str).str.lower().str.strip()
edges_df[to_col] = edges_df[to_col].astype(str).str.lower().str.strip()

# Remove self-loops
before = len(edges_df)
edges_df = edges_df[edges_df[from_col] != edges_df[to_col]].reset_index(drop=True)
print(f"  Removed {before - len(edges_df):,} self-loops")

# Remove rows with missing addresses
edges_df = edges_df.dropna(subset=[from_col, to_col]).reset_index(drop=True)

# Drop duplicates if hash column exists
if hash_col:
    before = len(edges_df)
    edges_df = edges_df.drop_duplicates(subset=[hash_col]).reset_index(drop=True)
    print(f"  Removed {before - len(edges_df):,} duplicate transactions")

# Value column: convert to float, fill NaN with 0
if value_col:
    edges_df[value_col] = pd.to_numeric(edges_df[value_col], errors='coerce').fillna(0).astype(np.float64)

print(f"\nCleaned dataset: {len(edges_df):,} transactions")
print(f"Unique senders: {edges_df[from_col].nunique():,}")
print(f"Unique receivers: {edges_df[to_col].nunique():,}")

---
## 2. Exploratory Data Analysis

In [ ]:
# Build address label mapping
address_labels = {}  # address -> label (0 or 1)

if label_col and label_col in edges_df.columns:
    # Labels are on transaction rows — propagate to addresses
    # Assume: if any tx involving an address is labeled fraud,
    # mark that sender address as fraud
    for _, row in edges_df[[from_col, label_col]].drop_duplicates().iterrows():
        addr = row[from_col]
        lbl = int(row[label_col])
        if lbl == 1:  # fraud/phishing
            address_labels[addr] = 1
        elif addr not in address_labels:
            address_labels[addr] = 0

elif labels_df is not None and label_col:
    # Labels in separate file
    addr_col = addr_col_in_labels or find_column(labels_df, FROM_COL_CANDIDATES + ['address'], required=True)
    labels_df[addr_col] = labels_df[addr_col].astype(str).str.lower().str.strip()
    for _, row in labels_df.iterrows():
        address_labels[row[addr_col]] = int(row[label_col])

n_fraud = sum(1 for v in address_labels.values() if v == 1)
n_legit = sum(1 for v in address_labels.values() if v == 0)

print(f"=== Label Summary ===")
print(f"  Labeled addresses: {len(address_labels):,}")
print(f"  Fraud/Phishing:    {n_fraud:,}")
print(f"  Legitimate:        {n_legit:,}")
print(f"  Fraud ratio:       {n_fraud/max(len(address_labels),1):.2%}")

In [ ]:
# EDA Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Label distribution
ax = axes[0]
all_addrs = set(edges_df[from_col]).union(set(edges_df[to_col]))
n_unlabeled = len(all_addrs) - len(address_labels)
categories = ['Fraud', 'Legitimate', 'Unlabeled']
counts = [n_fraud, n_legit, n_unlabeled]
colors = ['#c0392b', '#27ae60', '#95a5a6']
ax.bar(categories, counts, color=colors)
ax.set_ylabel('Count')
ax.set_title('Address Label Distribution')
for i, c in enumerate(counts):
    ax.text(i, c + max(counts)*0.01, f'{c:,}', ha='center', fontsize=10)

# 2. Transaction value distribution (if available)
ax = axes[1]
if value_col:
    vals = edges_df[value_col]
    vals_nonzero = vals[vals > 0]
    if len(vals_nonzero) > 0:
        ax.hist(np.log10(vals_nonzero + 1e-18), bins=100, color='#2980b9', alpha=0.7)
        ax.set_xlabel('log10(Value)')
    else:
        ax.hist(vals, bins=100, color='#2980b9', alpha=0.7)
        ax.set_xlabel('Value')
else:
    ax.text(0.5, 0.5, 'No value column', ha='center', va='center', transform=ax.transAxes)
ax.set_ylabel('Count')
ax.set_title('Transaction Value Distribution')

# 3. Transactions per address (degree distribution)
ax = axes[2]
degree_counts = edges_df[from_col].value_counts()
ax.hist(degree_counts.values, bins=100, color='#8e44ad', alpha=0.7, log=True)
ax.set_xlabel('Out-Degree (# transactions sent)')
ax.set_ylabel('Count (log)')
ax.set_title('Out-Degree Distribution')
ax.set_xlim(0, np.percentile(degree_counts.values, 99))

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Feature Engineering

Compute **12 node features** from the transaction graph.

| # | Feature | Description |
|---|---------|-------------|
| 1 | `in_degree` | Incoming transaction count |
| 2 | `out_degree` | Outgoing transaction count |
| 3 | `total_received` | Sum of incoming value |
| 4 | `total_sent` | Sum of outgoing value |
| 5 | `avg_value_in` | Mean incoming value |
| 6 | `avg_value_out` | Mean outgoing value |
| 7 | `max_tx_value` | Max single transaction |
| 8 | `unique_in_neighbors` | Distinct senders |
| 9 | `unique_out_neighbors` | Distinct receivers |
| 10 | `account_lifetime` | Last - first timestamp |
| 11 | `tx_frequency` | Transactions / lifetime |
| 12 | `in_out_ratio` | in_degree / total_degree |

In [ ]:
%%time

# Build node ID mapping
all_addresses = sorted(set(edges_df[from_col]).union(set(edges_df[to_col])))
node_to_id = {addr: idx for idx, addr in enumerate(all_addresses)}
num_nodes = len(all_addresses)
print(f"Total unique addresses (nodes): {num_nodes:,}")

# Use the value column if available, otherwise default to 1.0
has_value = value_col is not None
has_ts = ts_col is not None
print(f"Value column: {has_value}, Timestamp column: {has_ts}")

In [ ]:
%%time

# Compute node features via pandas groupby (faster than iteration)
print("Computing incoming features...")

# Map addresses to IDs in the dataframe
edges_df['_src_id'] = edges_df[from_col].map(node_to_id)
edges_df['_dst_id'] = edges_df[to_col].map(node_to_id)
edges_df['_value'] = edges_df[value_col].astype(np.float64) if has_value else 1.0

# --- Incoming features (grouped by receiver) ---
in_agg = edges_df.groupby('_dst_id').agg(
    in_degree=('_value', 'count'),
    total_received=('_value', 'sum'),
    avg_value_in=('_value', 'mean'),
    unique_in_neighbors=('_src_id', 'nunique'),
)

print("Computing outgoing features...")

# --- Outgoing features (grouped by sender) ---
out_agg = edges_df.groupby('_src_id').agg(
    out_degree=('_value', 'count'),
    total_sent=('_value', 'sum'),
    avg_value_out=('_value', 'mean'),
    max_tx_value=('_value', 'max'),
    unique_out_neighbors=('_dst_id', 'nunique'),
)

# --- Timestamp features (if available) ---
if has_ts:
    edges_df['_ts'] = pd.to_numeric(edges_df[ts_col], errors='coerce').fillna(0)
    ts_agg = edges_df.groupby('_src_id').agg(
        first_ts=('_ts', 'min'),
        last_ts=('_ts', 'max'),
    )
    ts_agg_in = edges_df.groupby('_dst_id').agg(
        first_ts_in=('_ts', 'min'),
        last_ts_in=('_ts', 'max'),
    )

print("Aggregation complete.")

In [ ]:
# Assemble 12-feature matrix
node_features = np.zeros((num_nodes, 12), dtype=np.float32)

# Fill from aggregations (indexed by node_id)
for nid in range(num_nodes):
    pass  # will vectorize below

# Vectorized fill
feat_df = pd.DataFrame(index=range(num_nodes))

feat_df['in_degree'] = 0.0
feat_df.loc[in_agg.index, 'in_degree'] = in_agg['in_degree'].values

feat_df['out_degree'] = 0.0
feat_df.loc[out_agg.index, 'out_degree'] = out_agg['out_degree'].values

feat_df['total_received'] = 0.0
feat_df.loc[in_agg.index, 'total_received'] = in_agg['total_received'].values

feat_df['total_sent'] = 0.0
feat_df.loc[out_agg.index, 'total_sent'] = out_agg['total_sent'].values

feat_df['avg_value_in'] = 0.0
feat_df.loc[in_agg.index, 'avg_value_in'] = in_agg['avg_value_in'].values

feat_df['avg_value_out'] = 0.0
feat_df.loc[out_agg.index, 'avg_value_out'] = out_agg['avg_value_out'].values

feat_df['max_tx_value'] = 0.0
feat_df.loc[out_agg.index, 'max_tx_value'] = out_agg['max_tx_value'].values

feat_df['unique_in_neighbors'] = 0.0
feat_df.loc[in_agg.index, 'unique_in_neighbors'] = in_agg['unique_in_neighbors'].values

feat_df['unique_out_neighbors'] = 0.0
feat_df.loc[out_agg.index, 'unique_out_neighbors'] = out_agg['unique_out_neighbors'].values

# Account lifetime
if has_ts:
    feat_df['account_lifetime'] = 0.0
    # Combine timestamps from both sender and receiver roles
    first_ts_all = pd.Series(np.inf, index=range(num_nodes))
    last_ts_all = pd.Series(-np.inf, index=range(num_nodes))
    if len(ts_agg) > 0:
        first_ts_all.loc[ts_agg.index] = np.minimum(first_ts_all.loc[ts_agg.index], ts_agg['first_ts'])
        last_ts_all.loc[ts_agg.index] = np.maximum(last_ts_all.loc[ts_agg.index], ts_agg['last_ts'])
    if len(ts_agg_in) > 0:
        first_ts_all.loc[ts_agg_in.index] = np.minimum(first_ts_all.loc[ts_agg_in.index], ts_agg_in['first_ts_in'])
        last_ts_all.loc[ts_agg_in.index] = np.maximum(last_ts_all.loc[ts_agg_in.index], ts_agg_in['last_ts_in'])
    lifetime = (last_ts_all - first_ts_all).clip(lower=0).replace([np.inf, -np.inf], 0)
    feat_df['account_lifetime'] = lifetime.values
else:
    feat_df['account_lifetime'] = 0.0

# Transaction frequency
total_degree = feat_df['in_degree'] + feat_df['out_degree']
lifetime_safe = feat_df['account_lifetime'].replace(0, np.nan)
feat_df['tx_frequency'] = (total_degree / lifetime_safe).fillna(0)

# In-out ratio
total_degree_safe = total_degree.replace(0, np.nan)
feat_df['in_out_ratio'] = (feat_df['in_degree'] / total_degree_safe).fillna(0.5)

# Convert to numpy
feature_names = [
    'in_degree', 'out_degree', 'total_received', 'total_sent',
    'avg_value_in', 'avg_value_out', 'max_tx_value',
    'unique_in_neighbors', 'unique_out_neighbors',
    'account_lifetime', 'tx_frequency', 'in_out_ratio'
]
node_features = feat_df[feature_names].values.astype(np.float32)

print(f"Feature matrix shape: {node_features.shape}")
print(pd.DataFrame(node_features, columns=feature_names).describe().to_string())

In [ ]:
# Normalize features
node_features = np.nan_to_num(node_features, nan=0.0, posinf=0.0, neginf=0.0)

scaler = StandardScaler()
node_features_scaled = scaler.fit_transform(node_features).astype(np.float32)

print(f"Scaled features: mean ~ {node_features_scaled.mean(axis=0).round(3)}")
print(f"Scaled features: std  ~ {node_features_scaled.std(axis=0).round(3)}")

In [ ]:
# Build label array: -1 = unlabeled, 0 = legitimate, 1 = fraud/phishing
labels = np.full(num_nodes, -1, dtype=np.int64)

for addr, lbl in address_labels.items():
    if addr in node_to_id:
        labels[node_to_id[addr]] = lbl

labeled_mask = labels >= 0
print(f"Labels assigned:")
print(f"  Fraud (1):      {(labels == 1).sum():,}")
print(f"  Legitimate (0): {(labels == 0).sum():,}")
print(f"  Unlabeled (-1): {(labels == -1).sum():,}")

---
## 4. Build DGL Graph

In [ ]:
%%time

# Build edge index (deduplicated directed edges)
edge_pairs = edges_df[['_src_id', '_dst_id']].drop_duplicates()
src_array = edge_pairs['_src_id'].values.astype(np.int64)
dst_array = edge_pairs['_dst_id'].values.astype(np.int64)

print(f"Edges (deduplicated): {len(src_array):,}")

# Construct DGL graph
dgl_graph = dgl.graph((src_array, dst_array), num_nodes=num_nodes)
dgl_graph.ndata['feat'] = torch.tensor(node_features_scaled, dtype=torch.float32)
dgl_graph.ndata['label'] = torch.tensor(labels, dtype=torch.long)

print(f"\nDGL Graph:")
print(f"  Nodes: {dgl_graph.num_nodes():,}")
print(f"  Edges: {dgl_graph.num_edges():,}")
print(f"  Features: {dgl_graph.ndata['feat'].shape}")

# Free memory
del edges_df, in_agg, out_agg, feat_df, edge_pairs
gc.collect()
print("DataFrame memory freed.")

---
## 5. Train / Validation / Test Split

In [ ]:
# Split labeled nodes: 60/20/20, stratified
labeled_indices = np.where(labeled_mask)[0]
labeled_labels = labels[labeled_indices]

train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    labeled_indices, labeled_labels,
    test_size=0.4, stratify=labeled_labels, random_state=42
)
val_idx, test_idx, val_labels, test_labels = train_test_split(
    temp_idx, temp_labels,
    test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_idx):,} (fraud: {(train_labels==1).sum()}, legit: {(train_labels==0).sum()})")
print(f"Val:   {len(val_idx):,} (fraud: {(val_labels==1).sum()}, legit: {(val_labels==0).sum()})")
print(f"Test:  {len(test_idx):,} (fraud: {(test_labels==1).sum()}, legit: {(test_labels==0).sum()})")

train_nids = torch.tensor(train_idx, dtype=torch.long)
val_nids = torch.tensor(val_idx, dtype=torch.long)
test_nids = torch.tensor(test_idx, dtype=torch.long)

# Class weights
n_legit_train = (train_labels == 0).sum()
n_fraud_train = (train_labels == 1).sum()
total_train = n_legit_train + n_fraud_train
w_legit = total_train / (2.0 * max(n_legit_train, 1))
w_fraud = total_train / (2.0 * max(n_fraud_train, 1))
class_weights = torch.tensor([w_legit, w_fraud], dtype=torch.float32).to(device)
print(f"\nClass weights: legit={w_legit:.4f}, fraud={w_fraud:.4f} ({w_fraud/w_legit:.1f}x)")

---
## 6. GraphSAGE Model

In [ ]:
class GraphSAGE(nn.Module):
    """2-layer GraphSAGE for node classification."""

    def __init__(self, in_feats, hidden_feats, out_feats, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_feats, hidden_feats, aggregator_type='mean')
        self.conv2 = SAGEConv(hidden_feats, out_feats, aggregator_type='mean')
        self.dropout = nn.Dropout(dropout)

    def forward(self, blocks, x):
        h = self.conv1(blocks[0], x)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(blocks[1], h)
        return h


# Hyperparameters
IN_FEATS = 12
HIDDEN_FEATS = 128
OUT_FEATS = 2
DROPOUT = 0.5
LR = 0.001
WEIGHT_DECAY = 5e-4
EPOCHS = 100
PATIENCE = 10
BATCH_SIZE = 1024
FANOUTS = [15, 10]

model = GraphSAGE(IN_FEATS, HIDDEN_FEATS, OUT_FEATS, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7. Training Loop

In [ ]:
sampler = NeighborSampler(FANOUTS)

train_dataloader = DataLoader(
    dgl_graph, train_nids, sampler,
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False
)
val_dataloader = DataLoader(
    dgl_graph, val_nids, sampler,
    batch_size=BATCH_SIZE, shuffle=False, drop_last=False
)

print(f"Train batches/epoch: {len(train_dataloader)}")
print(f"Val batches/epoch:   {len(val_dataloader)}")

In [ ]:
train_losses = []
val_f1_scores = []
best_val_f1 = 0.0
patience_counter = 0
best_model_state = None

print(f"Training for up to {EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(1, EPOCHS + 1):
    # === TRAIN ===
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for input_nodes, output_nodes, blocks in train_dataloader:
        blocks = [b.to(device) for b in blocks]
        feat = blocks[0].srcdata['feat']
        lbl = blocks[-1].dstdata['label']

        logits = model(blocks, feat)
        loss = F.cross_entropy(logits, lbl, weight=class_weights)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_loss)

    # === VALIDATE ===
    model.eval()
    val_preds, val_true = [], []

    with torch.no_grad():
        for input_nodes, output_nodes, blocks in val_dataloader:
            blocks = [b.to(device) for b in blocks]
            feat = blocks[0].srcdata['feat']
            lbl = blocks[-1].dstdata['label']
            preds = model(blocks, feat).argmax(dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(lbl.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average='binary', pos_label=1)
    val_f1_scores.append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1 or patience_counter == PATIENCE:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Val F1: {val_f1:.4f} | Best: {best_val_f1:.4f} | "
              f"Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best Val F1: {best_val_f1:.4f}")
        break

if best_model_state:
    model.load_state_dict(best_model_state)
    print("Restored best model weights.")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, color='#2980b9', linewidth=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_f1_scores, color='#27ae60', linewidth=1.5)
axes[1].axhline(y=best_val_f1, color='red', linestyle='--', alpha=0.5,
                label=f'Best F1: {best_val_f1:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Evaluation on Test Set

In [ ]:
model.eval()
test_dataloader = DataLoader(
    dgl_graph, test_nids, sampler,
    batch_size=BATCH_SIZE, shuffle=False, drop_last=False
)

test_preds, test_probs, test_true = [], [], []

with torch.no_grad():
    for input_nodes, output_nodes, blocks in test_dataloader:
        blocks = [b.to(device) for b in blocks]
        feat = blocks[0].srcdata['feat']
        lbl = blocks[-1].dstdata['label']
        logits = model(blocks, feat)
        probs = F.softmax(logits, dim=1)
        test_preds.extend(logits.argmax(dim=1).cpu().numpy())
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_true.extend(lbl.cpu().numpy())

test_preds = np.array(test_preds)
test_probs = np.array(test_probs)
test_true = np.array(test_true)

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print()
print(classification_report(
    test_true, test_preds,
    target_names=['Legitimate', 'Fraud/Phishing'],
    digits=4
))

test_f1 = f1_score(test_true, test_preds, pos_label=1)
test_auc_roc = roc_auc_score(test_true, test_probs)
test_auc_pr = average_precision_score(test_true, test_probs)

print(f"F1 Score (fraud):  {test_f1:.4f}")
print(f"AUC-ROC:           {test_auc_roc:.4f}")
print(f"AUC-PR:            {test_auc_pr:.4f}")

In [ ]:
# Evaluation plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(test_true, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(test_true, test_probs)
axes[1].plot(fpr, tpr, color='#2980b9', linewidth=2, label=f'AUC = {test_auc_roc:.4f}')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# PR Curve
prec, rec, _ = precision_recall_curve(test_true, test_probs)
axes[2].plot(rec, prec, color='#c0392b', linewidth=2, label=f'AUC-PR = {test_auc_pr:.4f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Save Model & Predictions

In [ ]:
OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

# Save model checkpoint
model_path = os.path.join(OUTPUT_DIR, 'graphsage_phishing.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'hyperparameters': {
        'in_feats': IN_FEATS, 'hidden_feats': HIDDEN_FEATS,
        'out_feats': OUT_FEATS, 'dropout': DROPOUT,
        'lr': LR, 'weight_decay': WEIGHT_DECAY,
        'fanouts': FANOUTS, 'batch_size': BATCH_SIZE,
    },
    'metrics': {
        'test_f1': test_f1, 'test_auc_roc': test_auc_roc,
        'test_auc_pr': test_auc_pr, 'best_val_f1': best_val_f1,
    },
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'class_weights': class_weights.cpu().numpy(),
    'feature_names': feature_names,
    'dataset': 'ethereum-transactions-for-fraud-detection (June 2024)',
}, model_path)
print(f"Model saved: {model_path} ({os.path.getsize(model_path)/1e6:.2f} MB)")

In [ ]:
# Save graph data
np.save(os.path.join(OUTPUT_DIR, 'node_features.npy'), node_features_scaled)
np.save(os.path.join(OUTPUT_DIR, 'edge_index.npy'), np.stack([src_array, dst_array]))
np.save(os.path.join(OUTPUT_DIR, 'labels.npy'), labels)

with open(os.path.join(OUTPUT_DIR, 'node_to_id.pkl'), 'wb') as f:
    pickle.dump(node_to_id, f)

print(f"Saved: node_features.npy {node_features_scaled.shape}")
print(f"Saved: edge_index.npy    {np.stack([src_array, dst_array]).shape}")
print(f"Saved: labels.npy        {labels.shape}")
print(f"Saved: node_to_id.pkl    {len(node_to_id):,} entries")

In [ ]:
# Save test predictions CSV
id_to_node = {v: k for k, v in node_to_id.items()}
test_results_df = pd.DataFrame({
    'node_id': test_idx,
    'address': [id_to_node[i] for i in test_idx],
    'true_label': test_true,
    'predicted_label': test_preds,
    'phishing_score': test_probs,
})
results_path = os.path.join(OUTPUT_DIR, 'test_predictions.csv')
test_results_df.to_csv(results_path, index=False)

print(f"\nTest predictions: {results_path}")
print(f"\nTop 10 highest-risk addresses:")
print(test_results_df.sort_values('phishing_score', ascending=False).head(10).to_string(index=False))

In [ ]:
# Final summary
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"")
print(f"  Dataset: Ethereum Transactions for Fraud Detection (Jun 2024)")
print(f"  Nodes:   {num_nodes:,}")
print(f"  Edges:   {len(src_array):,}")
print(f"  Features:{IN_FEATS}")
print(f"  Model:   GraphSAGE (2-layer, hidden={HIDDEN_FEATS})")
print(f"")
print(f"  Best Val F1:  {best_val_f1:.4f}")
print(f"  Test F1:      {test_f1:.4f}")
print(f"  Test AUC-ROC: {test_auc_roc:.4f}")
print(f"  Test AUC-PR:  {test_auc_pr:.4f}")
print(f"")
print(f"  Output files:")
for f in ['graphsage_phishing.pt', 'node_features.npy', 'edge_index.npy',
          'labels.npy', 'node_to_id.pkl', 'test_predictions.csv',
          'training_curves.png', 'evaluation_results.png', 'eda_distributions.png']:
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(fp):
        print(f"    {f:30s} {os.path.getsize(fp)/1e6:8.2f} MB")
print("=" * 60)